# Notebook 02 — Data Preprocessing and Encoding

## Objective

Prepare the Internet Firewall Data dataset for multiclass
machine learning classification.

This notebook implements duplicate handling, feature and
target separation, group-aware train/test splitting,
target encoding, feature transformations, scaling,
cross-validation preparation and class imbalance analysis.

The output is a reproducible training and testing dataset,
together with preprocessing configurations for model training.

In [20]:
import pandas as pd
import numpy as np

# Load the original dataset
df = pd.read_csv("../data/raw/log2.csv")

# Dataset dimensions
print("Dataset shape:", df.shape)

# Display the first five observations
display(df.head())

# Display column names and data types
print("\nData types:")
print(df.dtypes)

Dataset shape: (65532, 12)


,Source Port,Destination Port,NAT Source Port,NAT Destination Port,Action,Bytes,Bytes Sent,Bytes Received,Packets,Elapsed Time (sec),pkts_sent,pkts_received
0,57222,53,54587,53,allow,177,94,83,2,30,1,1
1,56258,3389,56258,3389,allow,4768,1600,3168,19,17,10,9
2,6881,50321,43265,50321,allow,238,118,120,2,1199,1,1
3,50553,3389,50553,3389,allow,3327,1438,1889,15,17,8,7
4,50002,443,45848,443,allow,25358,6778,18580,31,16,13,18



Data types:
Source Port             int64
Destination Port        int64
NAT Source Port         int64
NAT Destination Port    int64
Action                    str
Bytes                   int64
Bytes Sent              int64
Bytes Received          int64
Packets                 int64
Elapsed Time (sec)      int64
pkts_sent               int64
pkts_received           int64
dtype: object


## 2. Data Cleaning and Duplicate Handling

### Objective

Verify missing values and handle exact duplicate observations.

During EDA, we identified 8,362 duplicate records.

We remove exact duplicates to reduce redundancy and limit
the risk of duplicated observations affecting model evaluation.

The original dataset is preserved for reproducibility.

In [21]:
# Check missing values
print("Missing values:")
print(df.isnull().sum())

# Count exact duplicates
duplicates = df.duplicated().sum()

print("\nDuplicate rows:", duplicates)

# Remove exact duplicates
df_clean = df.drop_duplicates().copy()

# Reset row indices
df_clean = df_clean.reset_index(drop=True)

print("\nOriginal shape:", df.shape)
print("Cleaned shape:", df_clean.shape)

# Check target distribution after cleaning
print("\nClass distribution after cleaning:")
print(df_clean["Action"].value_counts())

Missing values:
Source Port             0
Destination Port        0
NAT Source Port         0
NAT Destination Port    0
Action                  0
Bytes                   0
Bytes Sent              0
Bytes Received          0
Packets                 0
Elapsed Time (sec)      0
pkts_sent               0
pkts_received           0
dtype: int64

Duplicate rows: 8362

Original shape: (65532, 12)
Cleaned shape: (57170, 12)

Class distribution after cleaning:
Action
allow         37439
drop          11635
deny           8042
reset-both       54
Name: count, dtype: int64


## 3. Feature and Target Separation

### Objective

Separate the input features (X) from the target variable (y).

- X contains the network traffic characteristics.
- y contains the firewall decision (Action).

This is a supervised multiclass classification problem.

The target variable must not be included among the
input features to avoid direct target leakage.

In [22]:
# Separate features and target

X = df_clean.drop(columns=["Action"])

y = df_clean["Action"]

# Verify dimensions
print("Features shape:", X.shape)
print("Target shape:", y.shape)

# Display feature names
print("\nInput features:")
print(X.columns.tolist())

# Display target classes
print("\nTarget classes:")
print(y.unique())

Features shape: (57170, 11)
Target shape: (57170,)

Input features:
['Source Port', 'Destination Port', 'NAT Source Port', 'NAT Destination Port', 'Bytes', 'Bytes Sent', 'Bytes Received', 'Packets', 'Elapsed Time (sec)', 'pkts_sent', 'pkts_received']

Target classes:
<ArrowStringArray>
['allow', 'drop', 'deny', 'reset-both']
Length: 4, dtype: str


## 4. Train/Test Split

### Objective

Divide the dataset into training and testing subsets.

- Training set: approximately 80% of observations.
- Testing set: approximately 20% of observations.

We use stratification to preserve class proportions,
especially for the minority class reset-both.

We also group observations with identical input features
to prevent the same feature combination from appearing
in both training and testing sets.

The test set will be reserved for final model evaluation.

In [23]:
from sklearn.model_selection import StratifiedGroupKFold

# Group identical input feature combinations
groups = X.groupby(
    list(X.columns),
    dropna=False,
    sort=False
).ngroup()

# Create five approximately stratified folds
splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Select a fold containing all four classes
splits = list(splitter.split(X, y, groups=groups))

valid_splits = [
    (train_idx, test_idx)
    for train_idx, test_idx in splits
    if y.iloc[train_idx].nunique() == 4
    and y.iloc[test_idx].nunique() == 4
]

if not valid_splits:
    raise ValueError("No split contains all four classes.")

train_idx, test_idx = valid_splits[0]

# Create training and testing sets
X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

print("\nTraining class distribution:")
print(y_train.value_counts())

print("\nTesting class distribution:")
print(y_test.value_counts())

Training shape: (45736, 11)
Testing shape: (11434, 11)

Training class distribution:
Action
allow         29951
drop           9308
deny           6434
reset-both       43
Name: count, dtype: int64

Testing class distribution:
Action
allow         7488
drop          2327
deny          1608
reset-both      11
Name: count, dtype: int64


## 5. Split Validation

### Objective

Verify that training and testing sets are correctly separated.

We check:

1. Dataset integrity.
2. Presence of all target classes.
3. Class distributions.
4. Absence of identical feature combinations
   across training and testing sets.

In [24]:
# Check total number of observations
print("Total observations:",
      len(X_train) + len(X_test))

# Check class distribution percentages
print("\nTraining distribution (%):")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting distribution (%):")
print(y_test.value_counts(normalize=True) * 100)

# Check for overlapping feature groups
train_groups = set(groups.iloc[train_idx])
test_groups = set(groups.iloc[test_idx])

overlap = train_groups.intersection(test_groups)

print("\nOverlapping feature groups:", len(overlap))

# Final validation
assert len(X_train) + len(X_test) == len(X)
assert len(overlap) == 0
assert set(y_train.unique()) == set(y_test.unique())

print("\nSplit validation successful!")

Total observations: 57170

Training distribution (%):
Action
allow         65.486706
drop          20.351583
deny          14.067693
reset-both     0.094018
Name: proportion, dtype: float64

Testing distribution (%):
Action
allow         65.488893
drop          20.351583
deny          14.063320
reset-both     0.096204
Name: proportion, dtype: float64

Overlapping feature groups: 0

Split validation successful!


## 6. Target Encoding

### Objective

Convert the categorical target variable into numerical labels.

We use LabelEncoder to encode firewall decisions.

The encoder is fitted on the training labels and
then applied to the testing labels.

The original class names are preserved through
the encoder's inverse_transform method.

In [25]:
from sklearn.preprocessing import LabelEncoder

# Initialize encoder
label_encoder = LabelEncoder()

# Fit using training labels
y_train_encoded = label_encoder.fit_transform(y_train)

# Transform testing labels
y_test_encoded = label_encoder.transform(y_test)

# Show encoding mapping
print("Class encoding:")

for i, action in enumerate(label_encoder.classes_):
    print(action, "->", i)

print("\nFirst 10 encoded training labels:")
print(y_train_encoded[:10])

Class encoding:
allow -> 0
deny -> 1
drop -> 2
reset-both -> 3

First 10 encoded training labels:
[0 0 0 0 0 0 0 0 0 0]


## 7. Feature Encoding and Scaling

### Objective

Prepare numerical input features for machine learning.

Traffic volume and duration features:
- Apply logarithmic transformation to reduce skewness.
- Apply StandardScaler to standardize their scales.

Port features:
- Treat port numbers as categorical identifiers.
- Apply OneHotEncoder to avoid assuming that numerical
  distances between port identifiers represent similarity.
- Group infrequent port values to control dimensionality.

All learned preprocessing operations must be fitted
on training data only.

In [26]:
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    FunctionTransformer
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Traffic measurements
traffic_features = [
    "Bytes",
    "Bytes Sent",
    "Bytes Received",
    "Packets",
    "Elapsed Time (sec)",
    "pkts_sent",
    "pkts_received"
]

# Port identifiers
port_features = [
    "Source Port",
    "Destination Port",
    "NAT Source Port",
    "NAT Destination Port"
]

# Verify that log1p is applicable
assert (X_train[traffic_features] >= 0).all().all()

# Traffic preprocessing
traffic_pipeline = Pipeline([
    ("log", FunctionTransformer(
    np.log1p,
    feature_names_out="one-to-one"
)),
    ("scaler", StandardScaler())
])

# Combine preprocessing steps
preprocessor = ColumnTransformer([
    ("traffic", traffic_pipeline, traffic_features),
    ("ports", OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=50
    ), port_features)
])

print("Preprocessing pipeline created!")

Preprocessing pipeline created!


## 8. Preprocessing Pipeline Verification

### Objective

Fit the preprocessing pipeline on training data only
and apply the learned transformations to the test set.

We verify:
- The transformed dataset dimensions.
- The consistency of feature dimensions.
- The absence of NaN or infinite values.

This transformation is performed for verification only.
Model training and cross-validation will use fresh
preprocessing pipelines to prevent data leakage.

In [27]:
from sklearn.base import clone
from scipy import sparse

# Create an independent copy for verification
preview_preprocessor = clone(preprocessor)

# Fit on training data only
X_train_preview = preview_preprocessor.fit_transform(X_train)

# Transform test data using learned parameters
X_test_preview = preview_preprocessor.transform(X_test)

print("Before preprocessing:")
print("Training:", X_train.shape)
print("Testing:", X_test.shape)

print("\nAfter preprocessing:")
print("Training:", X_train_preview.shape)
print("Testing:", X_test_preview.shape)

# Check consistency
assert X_train_preview.shape[1] == X_test_preview.shape[1]

# Check for invalid values
train_values = (
    X_train_preview.data
    if sparse.issparse(X_train_preview)
    else X_train_preview
)

test_values = (
    X_test_preview.data
    if sparse.issparse(X_test_preview)
    else X_test_preview
)

assert np.isfinite(train_values).all()
assert np.isfinite(test_values).all()

print("\nPreprocessing verification successful!")

Before preprocessing:
Training: (45736, 11)
Testing: (11434, 11)

After preprocessing:
Training: (45736, 80)
Testing: (11434, 80)

Preprocessing verification successful!


## 9. Transformed Feature Analysis

### Objective

Inspect the features generated by the preprocessing pipeline.

We examine:
- The number of generated features.
- The transformed feature names.
- The mean and standard deviation of scaled traffic features.

This step verifies that StandardScaler and OneHotEncoder
have been applied correctly.

In [28]:
# Get generated feature names
feature_names = preview_preprocessor.get_feature_names_out()

print("Original features:", X_train.shape[1])
print("Transformed features:", len(feature_names))

print("\nFirst 15 transformed features:")
print(feature_names[:15])

# Inspect scaled traffic features
scaled_traffic = (
    preview_preprocessor
    .named_transformers_["traffic"]
    .transform(X_train[traffic_features])
)

scaling_summary = pd.DataFrame({
    "Feature": traffic_features,
    "Mean": scaled_traffic.mean(axis=0),
    "Std": scaled_traffic.std(axis=0)
})

display(scaling_summary.round(4))

Original features: 11
Transformed features: 80

First 15 transformed features:
['traffic__Bytes' 'traffic__Bytes Sent' 'traffic__Bytes Received'
 'traffic__Packets' 'traffic__Elapsed Time (sec)' 'traffic__pkts_sent'
 'traffic__pkts_received' 'ports__Source Port_443'
 'ports__Source Port_3478' 'ports__Source Port_6881'
 'ports__Source Port_26900' 'ports__Source Port_27005'
 'ports__Source Port_37965' 'ports__Source Port_42298'
 'ports__Source Port_48817']


,Feature,Mean,Std
0,Bytes,-0.0,1.0
1,Bytes Sent,0.0,1.0
2,Bytes Received,-0.0,1.0
3,Packets,-0.0,1.0
4,Elapsed Time (sec),-0.0,1.0
5,pkts_sent,0.0,1.0
6,pkts_received,-0.0,1.0


## 10. Redundant Feature Investigation

### Objective

Investigate relationships between traffic measurement features.

We examine the following relationships:

- Bytes = Bytes Sent + Bytes Received
- Packets = pkts_sent + pkts_received

If these relationships hold consistently, some features
may be redundant.

We preserve the original features initially and will
evaluate feature removal during model comparison.

In [29]:
# Check byte relationships
bytes_consistency = (
    df_clean["Bytes"] ==
    df_clean["Bytes Sent"] + df_clean["Bytes Received"]
).mean()

# Check packet relationships
packets_consistency = (
    df_clean["Packets"] ==
    df_clean["pkts_sent"] + df_clean["pkts_received"]
).mean()

print("Bytes consistency:", bytes_consistency * 100, "%")
print("Packets consistency:", packets_consistency * 100, "%")

# Correlation between traffic features
correlation = df_clean[traffic_features].corr(
    method="spearman"
)

display(correlation.round(3))

Bytes consistency: 100.0 %
Packets consistency: 100.0 %


,Bytes,Bytes Sent,Bytes Received,Packets,Elapsed Time (sec),pkts_sent,pkts_received
Bytes,1.000,0.954,0.952,0.934,0.786,0.784,0.937
Bytes Sent,0.954,1.000,0.863,0.884,0.728,0.783,0.874
Bytes Received,0.952,0.863,1.000,0.940,0.794,0.764,0.976
Packets,0.934,0.884,0.940,1.000,0.799,0.873,0.968
Elapsed Time (sec),0.786,0.728,0.794,0.799,1.000,0.553,0.793
pkts_sent,0.784,0.783,0.764,0.873,0.553,1.000,0.806
pkts_received,0.937,0.874,0.976,0.968,0.793,0.806,1.000


## 11. Feature Sets for Leakage Analysis

### Objective

Prepare multiple feature sets to investigate the influence
of potentially problematic attributes.

Three scenarios are considered:

1. Full Features:
   All 11 input attributes.

2. Without NAT:
   Exclude NAT Source Port and NAT Destination Port.

3. Ports Only:
   Use Source Port and Destination Port as a restricted
   baseline.

These scenarios will be evaluated using identical
training and testing partitions.

The objective is to distinguish retrospective firewall
log classification from prediction using information
available before a firewall decision.

In [30]:
# Define experimental feature sets

feature_sets = {
    "full": X.columns.tolist(),

    "without_nat": [
        col for col in X.columns
        if col not in [
            "NAT Source Port",
            "NAT Destination Port"
        ]
    ],

    "ports_only": [
        "Source Port",
        "Destination Port"
    ]
}

# Display feature sets
for name, columns in feature_sets.items():
    print(f"\n{name}: {len(columns)} features")
    print(columns)

# Prepare training/testing subsets
feature_datasets = {}

for name, columns in feature_sets.items():
    feature_datasets[name] = {
        "X_train": X_train[columns].copy(),
        "X_test": X_test[columns].copy()
    }

print("\nFeature scenarios prepared successfully!")


full: 11 features
['Source Port', 'Destination Port', 'NAT Source Port', 'NAT Destination Port', 'Bytes', 'Bytes Sent', 'Bytes Received', 'Packets', 'Elapsed Time (sec)', 'pkts_sent', 'pkts_received']

without_nat: 9 features
['Source Port', 'Destination Port', 'Bytes', 'Bytes Sent', 'Bytes Received', 'Packets', 'Elapsed Time (sec)', 'pkts_sent', 'pkts_received']

ports_only: 2 features
['Source Port', 'Destination Port']

Feature scenarios prepared successfully!


## 12. Export Preprocessed Data

### Objective

Save the training and testing partitions for the
model training and evaluation phase.

We preserve:
- The original numerical feature values.
- The original firewall action labels.
- The selected train/test partition.
- The class encoding information.

Feature scaling and encoding will be performed inside
model pipelines during training and cross-validation.

This approach avoids preprocessing leakage.

In [31]:
from pathlib import Path
import json

# Create output directory
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Recombine features and target
train_data = X_train.copy()
train_data["Action"] = y_train

test_data = X_test.copy()
test_data["Action"] = y_test

# Save datasets
train_data.to_csv(output_dir / "train.csv", index=False)
test_data.to_csv(output_dir / "test.csv", index=False)

# Save reproducibility information
metadata = {
    "random_state": 42,
    "split_method": "StratifiedGroupKFold",
    "n_splits": 5,
    "training_rows": len(X_train),
    "testing_rows": len(X_test),
    "feature_sets": feature_sets,
    "label_mapping": {
        str(action): int(i)
        for i, action in enumerate(label_encoder.classes_)
    }
}

with open(output_dir / "split_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("Files saved successfully!")
print("Training observations:", len(train_data))
print("Testing observations:", len(test_data))
print("Output directory:", output_dir)

Files saved successfully!
Training observations: 45736
Testing observations: 11434
Output directory: ../data/processed


## 13. Export Validation

### Objective

Verify that the exported training and testing datasets
have been saved correctly.

We check:
- File existence
- Dataset dimensions
- Column consistency
- Presence of all four target classes

This ensures that Notebook 03 can load the datasets
without preprocessing inconsistencies.

In [32]:
# Reload exported datasets
train_loaded = pd.read_csv(output_dir / "train.csv")
test_loaded = pd.read_csv(output_dir / "test.csv")

# Check dimensions
print("Training dataset:", train_loaded.shape)
print("Testing dataset:", test_loaded.shape)

# Check column consistency
assert train_loaded.columns.tolist() == test_loaded.columns.tolist()

# Check dataset dimensions
assert train_loaded.shape == train_data.shape
assert test_loaded.shape == test_data.shape

# Check target classes
assert set(train_loaded["Action"].unique()) == set(label_encoder.classes_)
assert set(test_loaded["Action"].unique()) == set(label_encoder.classes_)

# Check missing values
assert train_loaded.isnull().sum().sum() == 0
assert test_loaded.isnull().sum().sum() == 0

print("\nExport validation successful!")

Training dataset: (45736, 12)
Testing dataset: (11434, 12)

Export validation successful!


## 14. Reusable Preprocessing Pipelines

### Objective

Create a reusable function that builds preprocessing
pipelines for different feature configurations.

Supported configurations:
- full
- without_nat
- ports_only

The function automatically identifies traffic features
and port features in the selected configuration.

This allows consistent preprocessing across different
machine learning experiments.

In [33]:
def make_preprocessor(feature_set="full"):

    # Selected features
    selected_features = feature_sets[feature_set]

    # Identify feature types
    numeric_cols = [
        col for col in traffic_features
        if col in selected_features
    ]

    port_cols = [
        col for col in port_features
        if col in selected_features
    ]

    transformers = []

    # Numerical transformations
    if numeric_cols:

        numeric_pipeline = Pipeline([
            ("log", FunctionTransformer(
                np.log1p,
                feature_names_out="one-to-one"
            )),
            ("scaler", StandardScaler())
        ])

        transformers.append(
            ("numeric", numeric_pipeline, numeric_cols)
        )

    # Port encoding
    if port_cols:

        port_encoder = OneHotEncoder(
            handle_unknown="infrequent_if_exist",
            min_frequency=50
        )

        transformers.append(
            ("ports", port_encoder, port_cols)
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )


# Test each preprocessing configuration
for name, columns in feature_sets.items():

    pipeline = make_preprocessor(name)

    X_sample = X_train[columns]

    transformed = pipeline.fit_transform(X_sample)

    print(
        name,
        "| Input features:", X_sample.shape[1],
        "| Output features:", transformed.shape[1]
    )

full | Input features: 11 | Output features: 80
without_nat | Input features: 9 | Output features: 61
ports_only | Input features: 2 | Output features: 54


## 15. Cross-Validation Preparation

### Objective

Prepare a stratified, group-aware cross-validation
strategy for model selection.

We use 5-fold StratifiedGroupKFold to:

- Approximately preserve class proportions.
- Prevent identical feature combinations from appearing
  in both training and validation folds.
- Evaluate models on multiple validation subsets.

The held-out test dataset is not used during
cross-validation or hyperparameter selection.

In [34]:
from sklearn.model_selection import StratifiedGroupKFold

# Group identical full feature combinations
cv_groups = X_train.groupby(
    list(X_train.columns),
    dropna=False,
    sort=False
).ngroup()

# Define cross-validation strategy
cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Inspect folds
for fold, (train_indices, val_indices) in enumerate(
    cv.split(X_train, y_train, groups=cv_groups),
    start=1
):

    y_val = y_train.iloc[val_indices]

    # Check group overlap
    groups_train = set(cv_groups.iloc[train_indices])
    groups_val = set(cv_groups.iloc[val_indices])

    assert len(groups_train & groups_val) == 0

    print(f"\nFold {fold}")
    print("Train size:", len(train_indices))
    print("Validation size:", len(val_indices))

    print("Validation class distribution:")
    print(
        y_val.value_counts().reindex(
            label_encoder.classes_,
            fill_value=0
        )
    )


Fold 1
Train size: 36589
Validation size: 9147
Validation class distribution:
Action
allow         5990
deny          1286
drop          1862
reset-both       9
Name: count, dtype: int64

Fold 2
Train size: 36589
Validation size: 9147
Validation class distribution:
Action
allow         5990
deny          1287
drop          1862
reset-both       8
Name: count, dtype: int64

Fold 3
Train size: 36588
Validation size: 9148
Validation class distribution:
Action
allow         5990
deny          1287
drop          1862
reset-both       9
Name: count, dtype: int64

Fold 4
Train size: 36589
Validation size: 9147
Validation class distribution:
Action
allow         5990
deny          1287
drop          1861
reset-both       9
Name: count, dtype: int64

Fold 5
Train size: 36589
Validation size: 9147
Validation class distribution:
Action
allow         5991
deny          1287
drop          1861
reset-both       8
Name: count, dtype: int64


## 16. Class Imbalance Analysis

### Objective

Investigate target class imbalance in the training set.

The dataset contains four classes:
- allow
- deny
- drop
- reset-both

The reset-both class is extremely rare.

We calculate class frequencies and candidate class
weights for later model training.

No oversampling or undersampling is applied at this stage
to avoid altering the original training distribution.

In [35]:
# Calculate training class distribution
class_counts = y_train.value_counts()

class_percentages = (
    y_train.value_counts(normalize=True) * 100
)

# Calculate balanced class weights
n_samples = len(y_train)
n_classes = y_train.nunique()

class_weights = (
    n_samples / (n_classes * class_counts)
)

# Create summary
imbalance_summary = pd.DataFrame({
    "Count": class_counts,
    "Percentage": class_percentages,
    "Balanced Weight": class_weights
})

display(imbalance_summary.round(4))

# Calculate imbalance ratio
imbalance_ratio = (
    class_counts.max() / class_counts.min()
)

print("\nImbalance ratio:", round(imbalance_ratio, 2))

,Count,Percentage,Balanced Weight
Action,,,
allow,29951,65.4867,0.3818
drop,9308,20.3516,1.2284
deny,6434,14.0677,1.7771
reset-both,43,0.0940,265.9070



Imbalance ratio: 696.53


## 17. Preprocessing Summary and Conclusions

### Completed Operations

1. Loaded the original Internet Firewall Data dataset.
2. Verified missing values and duplicate observations.
3. Removed exact duplicates for the primary experiment.
4. Separated input features and target variable.
5. Created stratified, group-aware train/test partitions.
6. Encoded the four firewall action classes.
7. Applied logarithmic transformation to traffic measurements.
8. Standardized numerical traffic features.
9. Applied frequency-aware One-Hot Encoding to port identifiers.
10. Verified the preprocessing pipeline.
11. Investigated mathematically redundant features.
12. Prepared alternative feature sets for leakage analysis.
13. Exported training/testing datasets and split metadata.
14. Prepared reusable pipelines and cross-validation.
15. Investigated the extreme target class imbalance.

### Main Findings

- The dataset contains 57,170 records after exact deduplication.
- Four firewall action classes are retained.
- Traffic measurements require scale-aware preprocessing.
- Port identifiers require appropriate categorical treatment.
- The reset-both class is extremely underrepresented.
- NAT attributes and post-connection statistics may introduce
  target leakage in a pre-decision prediction scenario.

### Next Milestone

Notebook 03: Model Training and Evaluation.

We will compare several classification algorithms using
reproducible cross-validation and appropriate multiclass metrics.

The held-out test set will be reserved for final evaluation
after model selection.

In [36]:
# Final milestone verification

assert len(X_train) + len(X_test) == len(df_clean)

assert X_train.shape[1] == 11
assert X_test.shape[1] == 11

assert len(label_encoder.classes_) == 4

assert (output_dir / "train.csv").exists()
assert (output_dir / "test.csv").exists()
assert (output_dir / "split_metadata.json").exists()

print("=" * 45)
print("MILESTONE 02 COMPLETED SUCCESSFULLY")
print("=" * 45)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Target classes:", label_encoder.classes_)
print("Feature scenarios:", list(feature_sets.keys()))

print("\nReady for Milestone 03!")

MILESTONE 02 COMPLETED SUCCESSFULLY
Training samples: 45736
Testing samples: 11434
Target classes: ['allow' 'deny' 'drop' 'reset-both']
Feature scenarios: ['full', 'without_nat', 'ports_only']

Ready for Milestone 03!
